In [13]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [14]:
PROJECT_ROOT = Path("..").resolve()

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "candidate_model.joblib"

print("Project root :", PROJECT_ROOT)
print("Train path   :", TRAIN_PATH)
print("Model path   :", MODEL_PATH)

print("\nTrain exists :", TRAIN_PATH.exists())
print("Model exists :", MODEL_PATH.exists())

Project root : C:\Users\Menaka\Videos\smart-food-demand-mlops
Train path   : C:\Users\Menaka\Videos\smart-food-demand-mlops\data\raw\train.csv
Model path   : C:\Users\Menaka\Videos\smart-food-demand-mlops\models\candidate_model.joblib

Train exists : True
Model exists : True


In [15]:
df = pd.read_csv(TRAIN_PATH)

df["date"] = pd.to_datetime(df["date"])

print("=" * 70)
print("DATASET")
print("=" * 70)

print("Rows   :", len(df))
print("Stores :", df["store"].nunique())
print("Store list:", sorted(df["store"].unique()))

print("\nDate range:")
print(df["date"].min(), "to", df["date"].max())

DATASET
Rows   : 5142
Stores : 9
Store list: ['store_0', 'store_1', 'store_2', 'store_3', 'store_4', 'store_5', 'store_6', 'store_7', 'store_8']

Date range:
2021-08-02 00:00:00 to 2023-11-30 00:00:00


In [16]:
def create_date_features(df):
    df = df.copy()

    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["day_of_week"] = df["date"].dt.dayofweek

    df["week_of_year"] = (
        df["date"]
        .dt.isocalendar()
        .week
        .astype(int)
    )

    df["quarter"] = df["date"].dt.quarter

    df["is_weekend"] = (
        df["day_of_week"] >= 5
    ).astype(int)

    return df


df = create_date_features(df)

print("Date features created.")

Date features created.


In [17]:
df = (
    df
    .sort_values(["store", "date"])
    .reset_index(drop=True)
)

df["sales_lag_1"] = (
    df
    .groupby("store")["sales"]
    .shift(1)
)

df["sales_lag_7"] = (
    df
    .groupby("store")["sales"]
    .shift(7)
)

df["sales_rolling_mean_7"] = (
    df
    .groupby("store")["sales"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(7)
         .mean()
    )
)

print("Historical features created.")

Historical features created.


In [18]:
SPLIT_DATE = pd.Timestamp("2023-06-16")

validation_df = df[
    df["date"] >= SPLIT_DATE
].copy()

validation_df = validation_df.dropna(
    subset=[
        "sales_lag_1",
        "sales_lag_7",
        "sales_rolling_mean_7"
    ]
)

print("=" * 70)
print("VALIDATION DATA")
print("=" * 70)

print("Rows:", len(validation_df))

print(
    "Date range:",
    validation_df["date"].min(),
    "to",
    validation_df["date"].max()
)

print(
    "\nStores:",
    validation_df["store"].nunique()
)

VALIDATION DATA
Rows: 1397
Date range: 2023-06-16 00:00:00 to 2023-11-30 00:00:00

Stores: 9


In [19]:
MODEL_FEATURES = [
    "store",
    "is_state_holiday",
    "is_school_holiday",
    "is_special_day",
    "temperature_max",
    "temperature_min",
    "temperature_mean",
    "sunshine_sum",
    "precipitation_sum",
    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "quarter",
    "is_weekend",
    "sales_lag_1",
    "sales_lag_7",
    "sales_rolling_mean_7"
]

X_validation = validation_df[MODEL_FEATURES]
y_validation = validation_df["sales"]

print("X shape:", X_validation.shape)
print("y shape:", y_validation.shape)

X shape: (1397, 19)
y shape: (1397,)


In [20]:
candidate_model = joblib.load(MODEL_PATH)

print("=" * 70)
print("MODEL")
print("=" * 70)

print("Model type:", type(candidate_model))
print("Loaded:", MODEL_PATH)

MODEL
Model type: <class 'sklearn.pipeline.Pipeline'>
Loaded: C:\Users\Menaka\Videos\smart-food-demand-mlops\models\candidate_model.joblib


In [21]:
predictions = candidate_model.predict(
    X_validation
)

overall_mae = mean_absolute_error(
    y_validation,
    predictions
)

overall_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        predictions
    )
)

overall_r2 = r2_score(
    y_validation,
    predictions
)

print("=" * 70)
print("9-STORE MODEL PERFORMANCE")
print("=" * 70)

print(f"MAE  : {overall_mae:.4f}")
print(f"RMSE : {overall_rmse:.4f}")
print(f"R²   : {overall_r2:.4f}")

9-STORE MODEL PERFORMANCE
MAE  : 0.2686
RMSE : 0.3475
R²   : 0.8001


In [22]:
validation_df["prediction"] = predictions

store_results = []

for store in sorted(
    validation_df["store"].unique()
):

    store_data = validation_df[
        validation_df["store"] == store
    ]

    actual = store_data["sales"]
    predicted = store_data["prediction"]

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    r2 = r2_score(
        actual,
        predicted
    )

    store_results.append({
        "store": store,
        "rows": len(store_data),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

store_metrics = pd.DataFrame(
    store_results
)

print("=" * 70)
print("PER-STORE PERFORMANCE")
print("=" * 70)

print(
    store_metrics
    .round(4)
    .to_string(index=False)
)

PER-STORE PERFORMANCE
  store  rows    MAE   RMSE      R2
store_0   167 0.2556 0.3088 -4.6047
store_1   168 0.2121 0.2853  0.7135
store_2    54 0.2197 0.2956 -0.1461
store_3   168 0.3174 0.3990 -1.1905
store_4   168 0.2079 0.2671  0.7105
store_5   168 0.3190 0.4022 -0.0757
store_6   168 0.2202 0.2732 -0.5292
store_7   168 0.3427 0.4395  0.3932
store_8   168 0.2893 0.3739  0.6700


In [23]:
store_profiles = (
    df
    .groupby("store")["sales"]
    .agg(
        mean_sales="mean",
        median_sales="median",
        std_sales="std",
        min_sales="min",
        max_sales="max",
        observations="count"
    )
    .reset_index()
)

print("=" * 70)
print("STORE DEMAND PROFILES")
print("=" * 70)

print(
    store_profiles
    .round(4)
    .to_string(index=False)
)

STORE DEMAND PROFILES
  store  mean_sales  median_sales  std_sales  min_sales  max_sales  observations
store_0     -1.3645       -1.4052     0.1629    -1.6337    -0.7336           273
store_1     -0.0329       -0.2112     0.8123    -1.6337     4.9865           847
store_2     -0.8463       -0.9248     0.2846    -1.2839    -0.1110            61
store_3     -0.7002       -0.7534     0.3516    -1.6430     0.6609           242
store_4      0.1884        0.0313     0.8451    -1.6453     4.3593           847
store_5      0.1696        0.0756     0.7242    -1.6057     3.0487           847
store_6     -1.0395       -1.1417     0.3882    -1.6570     1.5144           331
store_7      0.7872        0.7309     0.8900    -1.6244     7.5400           847
store_8      0.6961        0.5000     0.9721    -1.6104     6.8824           847


In [24]:
calibration = store_profiles[
    [
        "store",
        "median_sales",
        "mean_sales",
        "std_sales",
        "observations"
    ]
].copy()

calibration = calibration.rename(
    columns={
        "median_sales": "reference_demand"
    }
)

print("=" * 70)
print("INITIAL STORE CALIBRATION")
print("=" * 70)

print(
    calibration
    .round(4)
    .to_string(index=False)
)

INITIAL STORE CALIBRATION
  store  reference_demand  mean_sales  std_sales  observations
store_0           -1.4052     -1.3645     0.1629           273
store_1           -0.2112     -0.0329     0.8123           847
store_2           -0.9248     -0.8463     0.2846            61
store_3           -0.7534     -0.7002     0.3516           242
store_4            0.0313      0.1884     0.8451           847
store_5            0.0756      0.1696     0.7242           847
store_6           -1.1417     -1.0395     0.3882           331
store_7            0.7309      0.7872     0.8900           847
store_8            0.5000      0.6961     0.9721           847
